In [1]:
# 3 - modeling
# The purpose of this notebook is to make inference on new images using the current best model
#

In [2]:
# imports

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

import sys
import os
import importlib
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.build_metadata import build_df_from_data, build_df_from_new_data
from src.data import load_images_from_metadata, filter_images
from src.model import train_random_forest_model

In [3]:
# load all of the data into metadata csv file including the new data (data_new_2) and test data
from src.build_metadata import load_metadata
import importlib
import src.build_metadata as build_metadata

importlib.reload(build_metadata)
build_metadata.load_metadata(
    data_base_dir="../data",
    new_data_base_dir="../data_new",
    new_data_2_base_dir="../data_new_2",
    data_test_base_dir="../data_test",
    output_file="metadata.csv",
)

Applied human ratings to 852 DD rows.
Saved metadata to ../data\metadata.csv
Combined records: 1348


,LevelingScore,Person,DateCollected,ImageType,GSCamera,PanelID,State,FilePath
0,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (1...
1,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (2...
2,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (3...
3,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (4...
4,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (5...
...,...,...,...,...,...,...,...,...
1343,5.8,Brooke,7.21.26,TEST,2BDR-9F02,25-143,flat,../data_test\{leveling-5.8}_P{Brooke}_D{7.21.2...
1344,6.3,Brooke,7.21.26,TEST,2BDR-9F02,25-141,flat,../data_test\{leveling-6.3}_P{Brooke}_D{7.21.2...
1345,6.8,Brooke,7.21.26,TEST,2BDR-9F02,25-104,flat,../data_test\{leveling-6.8}_P{Brooke}_D{7.21.2...
1346,8.0,Brooke,7.21.26,TEST,2BDR-9F02,25-125,flat,../data_test\{leveling-8.0}_P{Brooke}_D{7.21.2...


In [4]:
# get the data
data = load_images_from_metadata("../data/metadata.csv", crop_fraction = .4, use_cv2=True)

data_standards = filter_images(data, ImageType = "STD", State = "flat", DateCollected = ["6.2.2026", "6.3.2026", "6.5.2026", "6.8.2026"])
data_real_paint = filter_images(data, ImageType = "DD", State = "flat")

Loaded 1348 images (cropping=ON, backend=cv2)
Filtered down to 177 images
Filtered down to 432 images


In [5]:
# current best model with the new data added (data_new_2) mae results reported after 10 epochs
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split, GroupKFold
from collections import Counter, defaultdict

torch.manual_seed(42)
np.random.seed(42)

# =========================
# ✅ DATA PREP
# =========================
def prepare_data_with_groups(data, img_size=(128, 128)):
    X, y, groups = [], [], []

    for d in data:
        img = d["image"].astype(np.float32) / 255.0
        img = torch.tensor(img).unsqueeze(0)

        img = torch.nn.functional.interpolate(
            img.unsqueeze(0), size=img_size,
            mode="bilinear", align_corners=False
        ).squeeze(0)

        X.append(img)
        y.append(d["LevelingScore"])
        groups.append(d.get("PanelID", None))

    return X, np.array(y), np.array(groups)

X_real, y_real, groups_real = prepare_data_with_groups(data_real_paint)
X_std, y_std, _ = prepare_data_with_groups(data_standards)

# =========================
# ✅ PANEL SPLIT
# =========================
unique_panels = np.unique(groups_real)

train_panels, test_panels = train_test_split(
    unique_panels, test_size=0.2, random_state=42
)

train_idx = np.isin(groups_real, train_panels)
test_idx  = np.isin(groups_real, test_panels)

X_train_real = [X_real[i] for i in range(len(X_real)) if train_idx[i]]
y_train_real = y_real[train_idx]
groups_train = groups_real[train_idx]

X_test = [X_real[i] for i in range(len(X_real)) if test_idx[i]]
y_test = y_real[test_idx]
groups_test = groups_real[test_idx]

# =========================
# ✅ TRANSFORMS
# =========================
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.RandomResizedCrop(128, scale=(0.95, 1.0)),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transform = transforms.Compose([
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# =========================
# ✅ MODEL
# =========================
class WaveletCNN(nn.Module):
    def __init__(self, num_blocks=5, base_channels=16):
        super().__init__()

        self.pool = nn.AvgPool2d(2)

        layers = []
        in_channels = 1
        channels = base_channels

        for i in range(num_blocks):
            layers += [
                nn.Conv2d(in_channels, channels, 3, padding=1),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
                nn.Conv2d(channels, channels, 3, padding=1),
                nn.ReLU()
            ]

            if i < num_blocks - 1:
                layers.append(nn.MaxPool2d(2))

            in_channels = channels
            channels *= 2

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        self.features = nn.Sequential(*layers)

        self.regressor = nn.Sequential(
            nn.Linear(in_channels + 3, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x_low1 = self.pool(x)
        x_low2 = self.pool(x_low1)

        m0 = x.mean(dim=[2,3])
        m1 = x_low1.mean(dim=[2,3])
        m2 = x_low2.mean(dim=[2,3])

        wavelet_feats = torch.cat([m0, m1, m2], dim=1)

        feats = self.features(x)
        feats = feats.view(feats.size(0), -1)

        combined = torch.cat([feats, wavelet_feats], dim=1)

        return self.regressor(combined)

# =========================
# ✅ SAMPLER
# =========================
def get_weighted_sampler(y):
    counter = Counter(y)
    weights = np.array([1.0 / counter[val] for val in y])
    return torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.float),
        num_samples=len(weights),
        replacement=True
    )

# =========================
# ✅ CROSS VALIDATION
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 16
epochs = 10

gkf = GroupKFold(n_splits=5)

cv_maes = []
cv_val_losses = []   # ✅ NEW

for fold, (train_idx_cv, val_idx_cv) in enumerate(
    gkf.split(X_train_real, y_train_real, groups_train)
):
    print(f"\n===== Fold {fold+1} =====")

    X_train_fold = [X_train_real[i] for i in train_idx_cv] + X_std
    y_train_fold = np.concatenate([y_train_real[train_idx_cv], y_std])

    X_val_fold = [X_train_real[i] for i in val_idx_cv]
    y_val_fold = y_train_real[val_idx_cv]

    sampler = get_weighted_sampler(y_train_fold)

    net = WaveletCNN().to(device)
    optimizer = optim.Adam(net.parameters(), lr=5e-4)
    criterion = nn.L1Loss()

    best_mae = float("inf")
    best_val_loss = float("inf")   # ✅ NEW

    for epoch in range(epochs):
        net.train()

        idx_list = list(sampler)

        for i in range(0, len(idx_list), batch_size):
            idx = idx_list[i:i+batch_size]

            imgs = torch.stack([
                train_transform(X_train_fold[j]) for j in idx
            ]).to(device)

            labels = torch.tensor(y_train_fold[idx]).float().to(device)

            optimizer.zero_grad()
            outputs = net(imgs).squeeze()

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # ✅ VALIDATION
        net.eval()
        X_val_proc = torch.stack(
            [test_transform(img) for img in X_val_fold]
        ).to(device)

        with torch.no_grad():
            outputs = net(X_val_proc).squeeze()
            preds = outputs.cpu().numpy()

        mae = np.mean(np.abs(y_val_fold - preds))

        labels = torch.tensor(y_val_fold).float().to(device)
        val_loss = criterion(outputs, labels).item()

# after epoch 10 finishes, mae and val_loss
# are the values from the LAST epoch

    print(f"Fold {fold+1} MAE (Epoch {epochs}): {mae:.3f}")

    cv_maes.append(mae)
    cv_val_losses.append(val_loss)

print("\nCV Mean MAE:", np.mean(cv_maes))
print("CV Mean Val Loss:", np.mean(cv_val_losses))   # ✅ NEW

# =========================
# ✅ FINAL TRAIN + TEST
# =========================
print("\n===== FINAL TRAIN + TEST =====")

X_train = X_train_real + X_std
y_train = np.concatenate([y_train_real, y_std])

sampler = get_weighted_sampler(y_train)

net = WaveletCNN().to(device)
optimizer = optim.Adam(net.parameters(), lr=5e-4)
criterion = nn.L1Loss()

for epoch in range(epochs):
    net.train()

    idx_list = list(sampler)

    for i in range(0, len(idx_list), batch_size):
        idx = idx_list[i:i+batch_size]

        imgs = torch.stack([
            train_transform(X_train[j]) for j in idx
        ]).to(device)

        labels = torch.tensor(y_train[idx]).float().to(device)

        optimizer.zero_grad()
        outputs = net(imgs).squeeze()

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# ✅ TEST
net.eval()

X_test_proc = torch.stack(
    [test_transform(img) for img in X_test]
).to(device)

with torch.no_grad():
    preds = net(X_test_proc).squeeze().cpu().numpy()

abs_error = np.abs(y_test - preds)

mae = np.mean(abs_error)
w1 = np.mean(abs_error <= 1) * 100
w2 = np.mean(abs_error <= 2) * 100

print(f"\nTEST → MAE={mae:.3f}, ±1={w1:.1f}%, ±2={w2:.1f}%")

# =========================
# ✅ CONSISTENCY METRIC
# =========================
panel_preds = defaultdict(list)

for pred, panel in zip(preds, groups_test):
    panel_preds[panel].append(pred)

stds = []

for panel in panel_preds:
    vals = np.array(panel_preds[panel])
    if len(vals) > 1:
        stds.append(np.std(vals))

mean_std = np.mean(stds)

print(f"Mean per-panel std: {mean_std:.3f}")


===== Fold 1 =====
Fold 1 MAE (Epoch 10): 0.616

===== Fold 2 =====
Fold 2 MAE (Epoch 10): 0.609

===== Fold 3 =====
Fold 3 MAE (Epoch 10): 0.942

===== Fold 4 =====
Fold 4 MAE (Epoch 10): 0.731

===== Fold 5 =====
Fold 5 MAE (Epoch 10): 1.184

CV Mean MAE: 0.8164543127203068
CV Mean Val Loss: 0.8164543390274048

===== FINAL TRAIN + TEST =====

TEST → MAE=0.657, ±1=76.8%, ±2=98.8%
Mean per-panel std: 0.317


In [6]:
# inference on data_test images using the trained model `net`
import pandas as pd

data_test = filter_images(data, ImageType = "TEST")

if len(data_test) == 0:
    print("No data_test images found in metadata. Re-run load_metadata after adding data_test rows.")
else:
    # Match model input prep used above: normalize to [0,1] and resize to 128x128.
    X_test_infer = []
    filepaths = []
    levels = []
    for d in data_test:
        img = d["image"].astype(np.float32) / 255.0
        img = torch.tensor(img).unsqueeze(0)
        img = torch.nn.functional.interpolate(
            img.unsqueeze(0), size=(128, 128), mode="bilinear", align_corners=False
        ).squeeze(0)
        X_test_infer.append(img)
        filepaths.append(d.get("FilePath"))
        levels.append(d.get("LevelingScore"))

    net.eval()
    X_test_infer_proc = torch.stack([test_transform(img) for img in X_test_infer]).to(device)

    with torch.no_grad():
        preds_test = net(X_test_infer_proc).view(-1).cpu().numpy()

    results = pd.DataFrame({
        "FilePath": filepaths,
        "TrueLevelingScore": levels,
        "PredictedLevelingScore": preds_test,
    })

    if results["TrueLevelingScore"].notna().all():
        abs_err = (results["TrueLevelingScore"].astype(float) - results["PredictedLevelingScore"]).abs()
        print(f"data_test MAE: {abs_err.mean():.3f}")

    display(results.head(20))
    results.to_csv("../data/data_test_predictions.csv", index=False)
    print("Saved predictions to ../data/data_test_predictions.csv")

Filtered down to 8 images
data_test MAE: 0.619


,FilePath,TrueLevelingScore,PredictedLevelingScore
0,..\data_test\{leveling-1.8}_P{Brooke}_D{7.21.2...,1.8,2.435251
1,..\data_test\{leveling-2.5}_P{Brooke}_D{7.21.2...,2.5,3.489389
2,..\data_test\{leveling-4.0}_P{Brooke}_D{7.21.2...,4.0,3.977562
3,..\data_test\{leveling-5.8}_P{Brooke}_D{7.21.2...,5.8,5.261693
4,..\data_test\{leveling-6.3}_P{Brooke}_D{7.21.2...,6.3,6.297597
5,..\data_test\{leveling-6.8}_P{Brooke}_D{7.21.2...,6.8,6.246617
6,..\data_test\{leveling-8.0}_P{Brooke}_D{7.21.2...,8.0,7.259971
7,..\data_test\{leveling-9.5}_P{Brooke}_D{7.21.2...,9.5,8.027054


Saved predictions to ../data/data_test_predictions.csv
